# Downloading traces from Logfire

Every conversation in the Streamlit app is captured in Logfire. Here we download the
traces and rebuild them as PydanticAI `AgentRunResult` objects, so we can use them for
evaluation later.

Requires `LOGFIRE_READ_TOKEN` in the project's `.env` (Logfire → project settings → Read tokens).

## Setting up the query client

In [15]:
import dotenv
dotenv.load_dotenv()

True

In [2]:
import os
from datetime import datetime, timedelta, timezone

from logfire.query_client import LogfireQueryClient

read_token = os.getenv('LOGFIRE_READ_TOKEN')
logfire_query_client = LogfireQueryClient(read_token=read_token)

# Newer Logfire versions require min_timestamp on every query: how far back to search
since = datetime.now(timezone.utc) - timedelta(days=30)

## Querying trace data

**Difference from the lesson:** the lesson's agent is a PydanticAI agent, so every run has an
`invoke_agent <name>` span. Our agent calls the OpenAI Responses API directly, so there is
no such span. Instead, one conversation = one `streamlit_session` trace (see `app.py`),
so we list those.

In [3]:
trace_rows = logfire_query_client.query_json_rows(
    min_timestamp=since,
    sql="""
    SELECT
        trace_id,
        start_timestamp,
        duration
    FROM records
    WHERE span_name = 'streamlit_session'
    ORDER BY start_timestamp DESC
    LIMIT 2
    """
)

In [4]:
trace_ids = [r['trace_id'] for r in trace_rows['rows']]
trace_ids

['01a1cb48798476e9729cb2329418f6f7', '01a1cb399c8c4bc3e9ee6fc3eeeb7bac']

For one trace, fetch the conversation data. In our traces it is not in
`pydantic_ai.all_messages` but on the OpenAI LLM-call spans (`streaming response from ...`):
each one has `gen_ai.input.messages`, `gen_ai.output.messages` and its own token usage.
We fetch all of them in order (`ASC`), because the session has one span per LLM call.

In [5]:
trace_id = trace_ids[0]

run_row = logfire_query_client.query_json_rows(
    min_timestamp=since,
    sql=f"""
    SELECT
        attributes->'gen_ai.input.messages' as input_messages,
        attributes->'gen_ai.output.messages' as output_messages,
        attributes->>'gen_ai.usage.input_tokens' as input_tokens,
        attributes->>'gen_ai.usage.output_tokens' as output_tokens
    FROM records
    WHERE trace_id = '{trace_id}'
      AND attributes->>'gen_ai.usage.input_tokens' IS NOT NULL
    ORDER BY start_timestamp ASC
    """
)

The raw data is in OpenTelemetry format (`role` / `parts` / `type`):

In [6]:
first_call = run_row['rows'][0]
first_call['input_messages'][0]

{'role': 'user',
 'parts': [{'type': 'text',
   'content': 'how many alexy g26 pro you have available?'}]}

## Converting back to PydanticAI format

`trace_replay` (copied from the course repo, adapted to our traces) does the reverse
conversion. `fetch_trace()` reads every LLM-call span of the trace, sums the token
usage and rebuilds the conversation.

In [7]:
from grigotel_ai_assistant import trace_replay

trace = trace_replay.fetch_trace(trace_id, logfire_query_client)
trace.all_messages[0]

{'role': 'system',
 'parts': [{'type': 'text',
   'content': 'You are the GrigoTel AI Assistant, a friendly sales and support assistant for GrigoTel.\nGrigoTel sells four smartphones: Alexy G26, Alexy G26 Fold, Alexy G26 Pro and Alexy G26 Lite.\n\n## Which tool to use\n- Technical specifications, features, setup, maintenance, battery, troubleshooting,\n  company information, privacy and compliance -> search_docs.\n  Pass `model` when the question is about a specific phone.\n- Prices, colors and availability -> get_product_info.\n  Never state a price or stock level from memory or from the documents.\n- Buying a phone -> follow the order process below.\n\n## Answering rules\n- Answer only from tool results. If the information is not there, say so. Never guess.\n- Always call a tool before answering any question about GrigoTel, its products or\n  its policies. Never answer from your own knowledge.\n- For troubleshooting and how-to questions, give only the steps in the documents.\n  Do no

In [8]:
run = trace_replay.trace_to_run_result(trace)
run.output

'{"answer":"The total price for **3 Alexy G26 Pro** in **Blue** is **2550 EUR**. \\n\\nWould you like to proceed with this order? Please confirm!","answer_type":"order","confidence":1.0,"found_in_sources":true,"sources":["GrigoTel database"],"follow_up_questions":["Yes, please place the order.","Can I order a different color?","What is the estimated delivery time?"]}'

`run.output` is the JSON text of our `AssistantResponse`. Parse it to get the typed object:

In [9]:
from grigotel_ai_assistant.agent import AssistantResponse

response = AssistantResponse.model_validate_json(run.output)
response.answer

Logfire project URL: https://logfire-eu.pydantic.dev/nima-dx/grigotel-ai-assistant

'The total price for **3 Alexy G26 Pro** in **Blue** is **2550 EUR**. \n\nWould you like to proceed with this order? Please confirm!'

In [10]:
# `usage` is a method in older PydanticAI versions and a property in newer ones
usage = run.usage if not callable(run.usage) else run.usage()
usage

RunUsage(input_tokens=17731, output_tokens=610)

## Fetching multiple traces and saving

In [11]:
traces = trace_replay.fetch_traces(trace_ids, logfire_query_client, since=since)

runs = []

for trace in traces.values():
    run = trace_replay.trace_to_run_result(trace)
    runs.append(run)

len(runs)

2

In [13]:
import pickle
from pathlib import Path

Path('../logfire_data').mkdir(exist_ok=True)

with open('../logfire_data/logs.bin', 'wb') as f_out:
    pickle.dump(runs, f_out)

To load them back later:

In [14]:
with open('../logfire_data/logs.bin', 'rb') as f_in:
    runs = pickle.load(f_in)

len(runs)

2